# Adım 1 — Ham Videolardan XY (17 bel üstü) + Açı (sağ/sol kol) .npy üretimi
- MediaPipe Pose ile her framede 2D landmark (normalize, [0..1]) çıkar.
- Bel üstü 17 nokta: yüz/omuz/kol + (referans için kalça merkezleri de açı hesabında kullanılacak).
- Açı seti (6): 
  - Sol kol: dirsek(LW-LE-LS), kol-gövde(LE-LS-hip), omuz hattı(LE-LS-midneck) 
  - Sağ kol: dirsek(RW-RE-RS), kol-gövde(RE-RS-hip), omuz hattı(RE-RS-midneck)
- Sonuçları .npy olarak kaydet.


## 1.1 – Yol & Klasörler

In [1]:
import os, glob, json
from pathlib import Path

BASE_ROOT   = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"
WORK_DIR    = os.path.join(BASE_ROOT, "ae_stajdevam")

TRAIN_VID_DIR = BASE_ROOT
TEST_VID_DIR  = os.path.join(BASE_ROOT, "test_video")

OUT_XY_TRAIN   = os.path.join(WORK_DIR, "npy_cikti")
OUT_XY_TEST    = os.path.join(WORK_DIR, "npy_test_cikti")
OUT_ANG_TRAIN  = os.path.join(WORK_DIR, "npy_aci_cikti")
OUT_ANG_TEST   = os.path.join(WORK_DIR, "npy_aci_test_cikti")

for d in [WORK_DIR, OUT_XY_TRAIN, OUT_XY_TEST, OUT_ANG_TRAIN, OUT_ANG_TEST]:
    os.makedirs(d, exist_ok=True)

print("Çıkış klasörleri hazır.")


Çıkış klasörleri hazır.


## 1.2 – Kurulum & İçe Aktarımlar

mediapipe ve opencv-python yoksa ilk satırları aç.

In [2]:
# !pip install mediapipe==0.10.9 opencv-python --quiet

import cv2
import numpy as np
import mediapipe as mp

mp_pose = mp.solutions.pose


## 1.3 – Üst gövde 17 nokta seçimi ve yardımcılar

Omuz-merkez ve omuz genişliği ileride normalizasyon için kritik.

XY .npy’de sadece 17 nokta tutulacak; açı hesabında gerektiğinde kalça mid-point (23/24) ayrıca okunur.

In [3]:
# MediaPipe Pose indeksleri (33 adet)
# Sık kullanılanlar:
#  0:nose, 11:L-shoulder, 12:R-shoulder, 13:L-elbow, 14:R-elbow,
#  15:L-wrist, 16:R-wrist, 23:L-hip, 24:R-hip
# Yüz için: 1..10 (göz/kaş/ağız bağlantıları)
UPPER17 = [0, 1,2,3,4,5,6, 7,8, 9,10, 11,12, 13,14, 15,16]  # toplam 17
LHIP, RHIP = 23, 24
LS, RS, LE, RE, LW, RW = 11, 12, 13, 14, 15, 16  # kısaltmalar

def _angle3(a, b, c):
    """a-b-c açısı (derece). a,b,c: (...,2)"""
    v1 = a - b; v2 = c - b
    v1 /= (np.linalg.norm(v1, axis=-1, keepdims=True) + 1e-8)
    v2 /= (np.linalg.norm(v2, axis=-1, keepdims=True) + 1e-8)
    cosv = np.clip(np.sum(v1*v2, axis=-1), -1.0, 1.0)
    return np.degrees(np.arccos(cosv))

def _fwd_fill_nan(arr):
    """(T,*) NaN'leri öne doğru doldur, başta hepsi NaN ise 0 kabul."""
    arr = arr.copy()
    mask = np.isnan(arr)
    if not mask.any():
        return arr
    # İlk geçerli değeri bul
    for t in range(arr.shape[0]):
        if not np.isnan(arr[t]).any():
            first = arr[t]
            arr[:t] = np.where(np.isnan(arr[:t]), first, arr[:t])
            break
    # İleri doldur
    for t in range(1, arr.shape[0]):
        arr[t] = np.where(np.isnan(arr[t]), arr[t-1], arr[t])
    # Hâlâ NaN kaldıysa 0 yap
    arr = np.nan_to_num(arr, nan=0.0)
    return arr


## 1.4 – Video’dan Landmark Çıkarma (XY) + Açı(6)

In [4]:
def extract_xy_and_angles(video_path: str):
    """
    Dönüş:
      xy_17: (T, 17, 2)  -> [0..1] normalize görüntü koordinatları
      ang6: (T, 6)       -> [deg] sonra /180 ile [0..1] ölçeğine alınabilir
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Açılamadı: {video_path}")

    xy_list = []
    ang_list = []

    with mp_pose.Pose(static_image_mode=False,
                      model_complexity=1,
                      enable_segmentation=False,
                      min_detection_confidence=0.5,
                      min_tracking_confidence=0.5) as pose:

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            # BGR->RGB ve inference
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)
            if not res.pose_landmarks:
                xy_list.append(np.full((len(UPPER17),2), np.nan, dtype=np.float32))
                ang_list.append(np.full((6,), np.nan, dtype=np.float32))
                continue

            lm = res.pose_landmarks.landmark  # 33 adet

            # XY 17 çıkar
            pts17 = []
            for idx in UPPER17:
                pts17.append([lm[idx].x, lm[idx].y])
            pts17 = np.array(pts17, dtype=np.float32)  # (17,2)

            # Açı için gerekli noktalar
            def _p(i): return np.array([lm[i].x, lm[i].y], dtype=np.float32)

            hip = (_p(LHIP) + _p(RHIP)) / 2.0
            midneck = (_p(LS) + _p(RS)) / 2.0

            # Sol kol (LW-LE-LS) dirsek, (LE-LS-hip) kol-gövde, (LE-LS-midneck) omuz hattı
            aL1 = _angle3(_p(LW), _p(LE), _p(LS))
            aL2 = _angle3(_p(LE), _p(LS), hip)
            aL3 = _angle3(_p(LE), _p(LS), midneck)

            # Sağ kol (RW-RE-RS), (RE-RS-hip), (RE-RS-midneck)
            aR1 = _angle3(_p(RW), _p(RE), _p(RS))
            aR2 = _angle3(_p(RE), _p(RS), hip)
            aR3 = _angle3(_p(RE), _p(RS), midneck)

            ang = np.array([aL1, aL2, aL3, aR1, aR2, aR3], dtype=np.float32)  # (6,)

            xy_list.append(pts17)
            ang_list.append(ang)

    cap.release()

    xy_17 = np.stack(xy_list, axis=0) if xy_list else np.zeros((0,17,2), np.float32)
    ang6  = np.stack(ang_list, axis=0) if ang_list else np.zeros((0,6), np.float32)

    # NaN temizliği (takip kaçırma anlarında olur)
    xy_17 = _fwd_fill_nan(xy_17)
    ang6  = _fwd_fill_nan(ang6)

    return xy_17, ang6


## 1.5 – Tek video işle ve kaydet

In [5]:
def process_and_save(video_path: str, out_xy_dir: str, out_ang_dir: str):
    name = Path(video_path).stem  # "1", "2", "61", "63"...
    xy, ang = extract_xy_and_angles(video_path)

    # Kaydet
    xy_path  = os.path.join(out_xy_dir,  f"{name}_xy.npy")
    ang_path = os.path.join(out_ang_dir, f"{name}_ang.npy")
    np.save(xy_path,  xy.astype(np.float32))
    np.save(ang_path, (ang/180.0).astype(np.float32))  # açıları [0,1] ölçekle
    return xy.shape, ang.shape, xy_path, ang_path


## 1.6 – 60 Eğitim + 3 Test videoyu sırayla işle

In [6]:
# Eğitim (1..60).mp4
train_videos = [os.path.join(TRAIN_VID_DIR, f"{i}.mp4") for i in range(1, 61)]
train_videos = [p for p in train_videos if os.path.exists(p)]

# Test (61,62,63).mp4
test_videos = [os.path.join(TEST_VID_DIR, f) for f in ["61.mp4","62.mp4","63.mp4"]]
test_videos = [p for p in test_videos if os.path.exists(p)]

print(f"Eğitim video sayısı: {len(train_videos)}  | Test video sayısı: {len(test_videos)}")

# Çalıştır
report = []
for vp in train_videos:
    shp_xy, shp_ang, xy_p, ang_p = process_and_save(vp, OUT_XY_TRAIN, OUT_ANG_TRAIN)
    report.append(("train", Path(vp).name, shp_xy, shp_ang))

for vp in test_videos:
    shp_xy, shp_ang, xy_p, ang_p = process_and_save(vp, OUT_XY_TEST, OUT_ANG_TEST)
    report.append(("test", Path(vp).name, shp_xy, shp_ang))

print("Tamamlandı. Örnek rapor ilk 5 satır:")
for r in report[:5]:
    print(r)


Eğitim video sayısı: 60  | Test video sayısı: 3
Tamamlandı. Örnek rapor ilk 5 satır:
('train', '1.mp4', (259, 17, 2), (259, 6))
('train', '2.mp4', (273, 17, 2), (273, 6))
('train', '3.mp4', (173, 17, 2), (173, 6))
('train', '4.mp4', (231, 17, 2), (231, 6))
('train', '5.mp4', (197, 17, 2), (197, 6))


## 1.7 – Hızlı kontrol

In [39]:
# Bir örnek yükleyip şekillere bakalım
sample_xy_files = glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy"))
sample_ang_files = glob.glob(os.path.join(OUT_ANG_TRAIN, "*_ang.npy"))
if sample_xy_files and sample_ang_files:
    A_xy = np.load(sample_xy_files[0])
    A_ang = np.load(sample_ang_files[0])
    print("XY shape:", A_xy.shape, "  (T,17,2)")
    print("ANG shape:", A_ang.shape, " (T,6)")
else:
    print("Henüz örnek bulunamadı. Birkaç videoyu işler misin?")


XY shape: (210, 17, 2)   (T,17,2)
ANG shape: (210, 6)  (T,6)


# Adım 2 — Pencereleme (windowing), pad, normalize ve PyTorch Dataset/DataLoader
- XY(17×2) + Açı(6) → her frame için 40 özellik
- XY normalizasyonu: omuz merkezi ile merkezle, omuz genişliğine göre ölçekle
- Pencereleme: window_len=30, stride=15 (pad: edge/zero/reflect)
- Çıkış tensörü: (C=40, T=30)
- Eğitim/Doğrulama ayrımı: video bazlı 80/20


## 2.1 – Konfig (window/pad/normalizasyon)

In [20]:
import os, glob, json, math
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Çalışma klasörleri (bir önceki adımla aynı)
BASE_ROOT   = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"
WORK_DIR    = os.path.join(BASE_ROOT, "ae_stajdevam")
OUT_XY_TRAIN   = os.path.join(WORK_DIR, "npy_cikti")
OUT_XY_TEST    = os.path.join(WORK_DIR, "npy_test_cikti")
OUT_ANG_TRAIN  = os.path.join(WORK_DIR, "npy_aci_cikti")
OUT_ANG_TEST   = os.path.join(WORK_DIR, "npy_aci_test_cikti")

STATS_JSON     = os.path.join(WORK_DIR, "feat_stats.json")

CFG = {
    "window_len": 30,
    "stride": 15,
    "pad_mode": "edge",        # "edge" | "zero" | "reflect"
    "use_angles": True,        # 6 açıyı dahil et
    "batch_size": 128,
    "num_workers": 0,          # Jupyter için 0 önerilir
    "seed": 42
}
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

print("CFG:", CFG)


CFG: {'window_len': 30, 'stride': 15, 'pad_mode': 'edge', 'use_angles': True, 'batch_size': 128, 'num_workers': 0, 'seed': 42}


## 2.2 – XY normalizasyonu ve özellik birleştirme (XY + Açı)

In [21]:
# UPPER17 sırasıyla kaydetmiştik: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]
# Bu dizide 11 -> L-Shoulder, 12 -> R-Shoulder pozisyon indexleri aynıdır (11 ve 12).
LS_POS, RS_POS = 11, 12

def normalize_xy_by_shoulders(xy_17: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    xy_17: (T,17,2)  [0..1] aralığında
    Omuz merkezine göre merkezle ve omuz genişliğine böl.
    Dönüş: (T,17,2)
    """
    c = (xy_17[:, LS_POS, :] + xy_17[:, RS_POS, :]) / 2.0              # (T,2)
    s = np.linalg.norm(xy_17[:, RS_POS, :] - xy_17[:, LS_POS, :], axis=1, keepdims=True)  # (T,1)
    s = np.maximum(s, eps)
    xy_n = (xy_17 - c[:, None, :]) / s[:, None, None]
    return xy_n.astype(np.float32)

def build_frame_features(xy_17: np.ndarray, ang6: np.ndarray, use_angles: bool = True) -> np.ndarray:
    """
    xy_17: (T,17,2), ang6: (T,6) [0..1] ölçekli
    Dönüş: (C, T)  -> C = 34 (+6) = 40
    """
    xy_n = normalize_xy_by_shoulders(xy_17)          # (T,17,2)
    xy_flat = xy_n.reshape(xy_n.shape[0], -1)        # (T,34)
    if use_angles and ang6 is not None and ang6.size > 0:
        feats = np.concatenate([xy_flat, ang6], axis=1)  # (T,40)
    else:
        feats = xy_flat                                   # (T,34)
    return feats.T.astype(np.float32)                     # (C,T)


## 2.3 – Pencereleme ve pad

In [22]:
def window_and_pad(x: np.ndarray, win: int, stride: int, mode: str = "edge") -> np.ndarray:
    """
    x: (C, T) → (Nwin, C, win)
    """
    C, T = x.shape
    out = []
    if T <= 0:
        return np.zeros((1, C, win), dtype=x.dtype)

    for start in range(0, max(1, T - win + 1), stride):
        end = start + win
        if end <= T:
            out.append(x[:, start:end])
        else:
            deficit = end - T
            if mode == "zero":
                pad = np.zeros((C, deficit), dtype=x.dtype)
            elif mode == "reflect":
                take = min(deficit, T)
                pad = np.flip(x[:, T - take:T], axis=1)
                if take < deficit:
                    pad = np.concatenate([pad, np.repeat(x[:, :1], deficit - take, axis=1)], axis=1)
            else:  # "edge"
                pad = np.repeat(x[:, -1:], deficit, axis=1)
            out.append(np.concatenate([x[:, start:T], pad], axis=1))

    if not out:  # T < win, hiç döngüye girmediyse
        deficit = win - T
        if mode == "zero":
            pad = np.zeros((C, deficit), dtype=x.dtype)
        elif mode == "reflect":
            take = min(deficit, T)
            pad = np.flip(x[:, T - take:T], axis=1)
            if take < deficit:
                pad = np.concatenate([pad, np.repeat(x[:, :1], deficit - take, axis=1)], axis=1)
        else:
            pad = np.repeat(x[:, -1:], deficit, axis=1)
        out = [np.concatenate([x, pad], axis=1)]
    return np.stack(out, axis=0)  # (Nwin, C, win)


## 2.4 – Eğitim seti için özellik istatistikleri (mean/std) çıkar ve kaydet

In [23]:
def compute_and_save_feature_stats(xy_dir: str, ang_dir: str, stats_json: str, cfg: dict):
    """
    Tüm eğitim videolarını dolaşarak (frame bazlı) ortalama ve std hesaplar.
    Not: Önce frame özellikleri -> sonra (C,) mean/std
    """
    xy_files = sorted(glob.glob(os.path.join(xy_dir, "*_xy.npy")))
    assert xy_files, f"Eğitim XY bulunamadı: {xy_dir}"
    sums = None
    sums2 = None
    count = 0

    for xy_path in xy_files:
        base = Path(xy_path).name.replace("_xy.npy", "")
        ang_path = os.path.join(ang_dir, f"{base}_ang.npy")
        xy = np.load(xy_path)            # (T,17,2)
        ang = np.load(ang_path) if os.path.exists(ang_path) else None  # (T,6) [0..1]

        feat = build_frame_features(xy, ang, cfg["use_angles"])  # (C,T)
        C, T = feat.shape
        if sums is None:
            sums  = np.zeros(C, dtype=np.float64)
            sums2 = np.zeros(C, dtype=np.float64)

        sums  += feat.mean(axis=1) * T    # ortalamanın T ile ağırlığı
        sums2 += (feat**2).mean(axis=1) * T
        count += T

    mean = (sums / count).astype(np.float32)
    var  = (sums2 / count) - (mean.astype(np.float64)**2)
    std  = np.sqrt(np.maximum(var, 1e-8)).astype(np.float32)

    stats = {"mean": mean.tolist(), "std": std.tolist(), "C": int(len(mean))}
    with open(stats_json, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2)
    print(f"Kaydedildi: {stats_json}")
    return mean, std

def load_feature_stats(stats_json: str):
    with open(stats_json, "r", encoding="utf-8") as f:
        stats = json.load(f)
    mean = np.array(stats["mean"], dtype=np.float32)
    std  = np.array(stats["std"], dtype=np.float32)
    return mean, std


## 2.5 – Global indeksleme ile PyTorch Dataset

In [24]:
def count_windows(T: int, win: int, stride: int) -> int:
    if T <= 0:
        return 1
    if T <= win:
        return 1
    return 1 + math.ceil((T - win) / stride)

class PoseWindowDataset(Dataset):
    """
    Global indeks → (file_idx, start_index) haritalı.
    __getitem__ dönünce: (C,win) tensörü.
    """
    def __init__(self, xy_dir: str, ang_dir: str, stats_json: str, cfg: dict, file_list=None):
        self.xy_dir = xy_dir
        self.ang_dir = ang_dir
        self.cfg = cfg

        all_xy = sorted(glob.glob(os.path.join(xy_dir, "*_xy.npy")))
        if file_list is not None:
            # file_list: dosya baz adları (örn: "1", "2", ...)
            all_xy = [os.path.join(xy_dir, f"{b}_xy.npy") for b in file_list]
        assert all_xy, f"Boş dizin: {xy_dir}"

        self.files = []
        self.index = []   # (file_idx, start_frame)
        win, stride = cfg["window_len"], cfg["stride"]

        # stats yükle
        self.mean, self.std = load_feature_stats(stats_json)
        self.C = len(self.mean)

        for xy_path in all_xy:
            base = Path(xy_path).name.replace("_xy.npy", "")
            ang_path = os.path.join(ang_dir, f"{base}_ang.npy")
            xy = np.load(xy_path)
            ang = np.load(ang_path) if os.path.exists(ang_path) else None

            feats = build_frame_features(xy, ang, cfg["use_angles"])  # (C,T)
            T = feats.shape[1]
            nwin = count_windows(T, win, stride)
            self.files.append((xy_path, ang_path, T))

            # start indexleri (frame bazlı) sadece referans, gerçek slice window_and_pad ile
            for k in range(nwin):
                start = min(k*stride, max(0, T - win))
                self.index.append((len(self.files)-1, start))

        print(f"Toplam video: {len(self.files)} | Toplam pencere: {len(self.index)}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        file_idx, start = self.index[idx]
        xy_path, ang_path, T = self.files[file_idx]
        xy = np.load(xy_path)
        ang = np.load(ang_path) if os.path.exists(ang_path) else None

        xCT = build_frame_features(xy, ang, self.cfg["use_angles"])  # (C,T)
        # Standardizasyon
        xCT = (xCT - self.mean[:, None]) / (self.std[:, None] + 1e-8)
        # Pencere & pad
        win = self.cfg["window_len"]; stride = self.cfg["stride"]
        stop = start + win
        if stop <= xCT.shape[1]:
            xwin = xCT[:, start:stop]
        else:
            # tek örnek için pad
            deficit = stop - xCT.shape[1]
            if self.cfg["pad_mode"] == "zero":
                pad = np.zeros((xCT.shape[0], deficit), dtype=xCT.dtype)
            elif self.cfg["pad_mode"] == "reflect":
                take = min(deficit, xCT.shape[1])
                pad = np.flip(xCT[:, -take:], axis=1)
                if take < deficit:
                    pad = np.concatenate([pad, np.repeat(xCT[:, -1:], deficit - take, axis=1)], axis=1)
            else:
                pad = np.repeat(xCT[:, -1:], deficit, axis=1)
            xwin = np.concatenate([xCT[:, start:], pad], axis=1)

        x = torch.from_numpy(xwin.astype(np.float32))   # (C,win)
        return x


## 2.6 – Eğitim/Doğrulama ayrımı ve DataLoader’lar

In [47]:
# ==== 2.7 — Eğitim/Doğrulama ayrımı + İstatistik + Dataset/DataLoader (temiz & bağımsız) ====
import os, glob, json, math
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# ------------------ Yollar & Ayarlar ------------------
BASE_ROOT = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"
WORK_DIR  = os.path.join(BASE_ROOT, "ae_stajdevam")
OUT_XY_TRAIN  = os.path.join(WORK_DIR, "npy_cikti")
OUT_ANG_TRAIN = os.path.join(WORK_DIR, "npy_aci_cikti")
STATS_JSON    = os.path.join(WORK_DIR, "feat_stats.json")

CFG = {"window_len":30, "stride":15, "pad_mode":"edge",
       "use_angles":True, "batch_size":128, "num_workers":0, "seed":42}
np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

# ------------------ Yardımcılar (bu hücreye özel) ------------------
LS_POS, RS_POS = 11, 12  # L-Shoulder, R-Shoulder

def _normalize_xy_by_shoulders(xy_17: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    # xy_17: (T,17,2) -> omuz merkezle/ölçekle
    xy_17 = np.asarray(xy_17, dtype=np.float32)
    assert xy_17.ndim == 3 and xy_17.shape[1:] == (17,2), f"XY (T,17,2) olmalı, geldi {xy_17.shape}"
    T = xy_17.shape[0]
    # center (T,2), scale (T,1)
    c = (xy_17[:, LS_POS, :] + xy_17[:, RS_POS, :]) / 2.0
    s = np.linalg.norm(xy_17[:, RS_POS, :] - xy_17[:, LS_POS, :], axis=1, keepdims=True)
    s = np.maximum(s, eps)
    # force explicit shapes for safe broadcasting
    c = c.reshape(T, 1, 2)   # (T,1,2)
    s = s.reshape(T, 1, 1)   # (T,1,1)
    return ((xy_17 - c) / s).astype(np.float32)

def _make_features_CT(xy_17: np.ndarray, ang6: np.ndarray | None, use_angles=True) -> np.ndarray:
    """
    Girdi: xy_17 (T,17,2), ang6 (T,6) veya None
    Çıktı: (C,T) -> C=40 (34 XY + 6 açı)
    """
    xy_17 = np.asarray(xy_17, dtype=np.float32)
    assert xy_17.ndim == 3 and xy_17.shape[1:] == (17,2), f"XY (T,17,2) olmalı, geldi {xy_17.shape}"
    T = xy_17.shape[0]

    if ang6 is None:
        ang6 = np.zeros((T,6), dtype=np.float32)
    else:
        ang6 = np.asarray(ang6, dtype=np.float32)
        if ang6.ndim == 1 and ang6.size % 6 == 0:
            ang6 = ang6.reshape(-1,6)
        if ang6.shape == (6, T):
            ang6 = ang6.T
        assert ang6.ndim == 2 and ang6.shape[1] == 6, f"Açı (T,6) olmalı, geldi {ang6.shape}"
        if ang6.shape[0] != T:
            out = np.zeros((T,6), np.float32)
            tmin = min(T, ang6.shape[0])
            out[:tmin] = ang6[:tmin]
        ang6 = out if 'out' in locals() else ang6

    xy_n    = _normalize_xy_by_shoulders(xy_17)  # (T,17,2)
    xy_flat = xy_n.reshape(T, 34)                # (T,34)
    feats_TxC = np.concatenate([xy_flat, ang6], axis=1) if use_angles else xy_flat  # (T,40/34)
    feats_CxT = feats_TxC.T.astype(np.float32)  # (C,T)
    if use_angles:
        assert feats_CxT.shape[0] == 40, f"Kanal 40 olmalı, geldi {feats_CxT.shape[0]}"
    else:
        assert feats_CxT.shape[0] == 34, f"Kanal 34 olmalı, geldi {feats_CxT.shape[0]}"
    return feats_CxT

def _window_slice_or_pad(xCT: np.ndarray, start: int, win: int, mode: str="edge") -> np.ndarray:
    C, T = xCT.shape
    end = start + win
    if end <= T:
        return xCT[:, start:end]
    deficit = end - T
    if mode == "zero":
        pad = np.zeros((C, deficit), dtype=xCT.dtype)
    elif mode == "reflect":
        take = min(deficit, T)
        pad = np.flip(xCT[:, -take:], axis=1)
        if take < deficit:
            pad = np.concatenate([pad, np.repeat(xCT[:, -1:], deficit - take, axis=1)], axis=1)
    else:  # edge
        pad = np.repeat(xCT[:, -1:], deficit, axis=1)
    return np.concatenate([xCT[:, start:], pad], axis=1)

def _load_feature_stats(stats_json: str):
    with open(stats_json, "r", encoding="utf-8") as f:
        s = json.load(f)
    mean = np.array(s["mean"], np.float32)
    std  = np.array(s["std"],  np.float32)
    return mean, std

# ------------------ 1) İstatistik: yoksa hesapla ------------------
if not os.path.exists(STATS_JSON):
    xy_files = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))
    assert xy_files, f"Eğitim XY bulunamadı: {OUT_XY_TRAIN}"
    sums  = np.zeros(40, np.float64); sums2 = np.zeros(40, np.float64); count = 0
    for p in xy_files:
        base  = Path(p).name.replace("_xy.npy", "")
        p_ang = os.path.join(OUT_ANG_TRAIN, f"{base}_ang.npy")
        xy = np.load(p)                                     # (T,17,2)
        ang = np.load(p_ang) if os.path.exists(p_ang) else None  # (T,6) veya None
        feat = _make_features_CT(xy, ang, CFG["use_angles"])     # (40,T)
        sums  += feat.sum(axis=1)
        sums2 += (feat**2).sum(axis=1)
        count += feat.shape[1]
    mean = (sums / count).astype(np.float32)
    var  = (sums2 / count) - (mean.astype(np.float64)**2)
    std  = np.sqrt(np.maximum(var, 1e-8)).astype(np.float32)
    with open(STATS_JSON, "w", encoding="utf-8") as f:
        json.dump({"mean":mean.tolist(), "std":std.tolist(), "C":40}, f, indent=2)
print("feat_stats.json:", STATS_JSON, "->", os.path.exists(STATS_JSON))

# ------------------ 2) Split (80/20 video bazlı) ------------------
all_train_xy = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))
assert all_train_xy, f"Eğitim XY bulunamadı: {OUT_XY_TRAIN}"
base_names = [Path(p).name.replace("_xy.npy", "") for p in all_train_xy]
base_names = sorted(base_names, key=lambda s: int(Path(s).stem) if Path(s).stem.isdigit() else s)
n = len(base_names); n_train = max(1, int(0.8*n))
train_list = base_names[:n_train]
val_list   = base_names[n_train:] if n_train < n else base_names[-1:]  # en az 1 val
print(f"Video bazlı split -> Train: {len(train_list)} | Val: {len(val_list)} | Toplam: {n}")

# ------------------ 3) Dataset ------------------
class PoseWindowDataset(Dataset):
    def __init__(self, xy_dir: str, ang_dir: str, stats_json: str, cfg: dict, file_list=None):
        self.cfg = cfg
        self.mean, self.std = _load_feature_stats(stats_json)
        xy_files = sorted(glob.glob(os.path.join(xy_dir, "*_xy.npy")))
        if file_list is not None:
            xy_files = [os.path.join(xy_dir, f"{b}_xy.npy") for b in file_list]
        assert xy_files, f"Boş dizin: {xy_dir}"

        self.items = []  # (feat(40,T), base)
        self.index = []  # (i, start)
        for p in xy_files:
            base  = Path(p).name.replace("_xy.npy", "")
            p_ang = os.path.join(ang_dir, f"{base}_ang.npy")
            xy = np.load(p)
            ang = np.load(p_ang) if os.path.exists(p_ang) else None
            feat = _make_features_CT(xy, ang, cfg["use_angles"])  # (40,T)
            self.items.append((feat, base))

            T = feat.shape[1]; win = cfg["window_len"]; stride = cfg["stride"]
            if T <= win:
                self.index.append((len(self.items)-1, 0))
            else:
                for st in range(0, T - win + 1, stride):
                    self.index.append((len(self.items)-1, st))
                if (T - win) % stride != 0:
                    self.index.append((len(self.items)-1, T - win))
        print(f"Toplam video: {len(self.items)} | Toplam pencere: {len(self.index)}")

    def __len__(self): return len(self.index)

    def __getitem__(self, idx):
        i, start = self.index[idx]
        feat, _ = self.items[i]  # (40,T)
        x = _window_slice_or_pad(feat, start, self.cfg["window_len"], self.cfg["pad_mode"])
        # standardizasyon
        x = (x - self.mean[:,None]) / (self.std[:,None] + 1e-8)
        return torch.from_numpy(x.astype(np.float32))  # (40,win)

# ------------------ 4) DataLoader'lar ------------------
train_ds = PoseWindowDataset(OUT_XY_TRAIN, OUT_ANG_TRAIN, STATS_JSON, CFG, file_list=train_list)
val_ds   = PoseWindowDataset(OUT_XY_TRAIN, OUT_ANG_TRAIN, STATS_JSON, CFG, file_list=val_list)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], drop_last=False)

# ------------------ 5) Hızlı kontrol ------------------
xb = next(iter(train_loader))
print("Bir batch şekli:", tuple(xb.shape))  # (B, 40, 30) beklenir


feat_stats.json: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\feat_stats.json -> True
Video bazlı split -> Train: 48 | Val: 12 | Toplam: 60
Toplam video: 48 | Toplam pencere: 624
Toplam video: 12 | Toplam pencere: 140
Bir batch şekli: (128, 40, 30)


In [48]:
import glob, os, numpy as np
print('OUT_XY_TRAIN =', OUT_XY_TRAIN)
xy_files = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))
print('found', len(xy_files), 'xy files')
for i,p in enumerate(xy_files[:8]):
    print(i, p)
    try:
        a = np.load(p)
        print('  -> shape', a.shape, 'dtype', a.dtype, 'size', a.size, 'ndim', a.ndim)
    except Exception as e:
        print('  -> load error', e)

if 'p' in globals():
    print('\nlast p variable from kernel:', p)
    try:
        a = np.load(p)
        print('  last p shape', a.shape, 'dtype', a.dtype, 'size', a.size, 'ndim', a.ndim)
    except Exception as e:
        print('  error loading last p:', e)


OUT_XY_TRAIN = C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti
found 60 xy files
0 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\10_xy.npy
  -> shape (210, 17, 2) dtype float32 size 7140 ndim 3
1 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\11_xy.npy
  -> shape (195, 17, 2) dtype float32 size 6630 ndim 3
2 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\12_xy.npy
  -> shape (214, 17, 2) dtype float32 size 7276 ndim 3
3 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\13_xy.npy
  -> shape (222, 17, 2) dtype float32 size 7548 ndim 3
4 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\14_xy.npy
  -> shape (239, 17, 2) dtype float32 size 8126 ndim 3
5 C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\15_xy.npy
  -> shape (194, 17, 2) dtype float32 size 6596 ndim 3
6 C:\Users\The Coder Farmer\Desktop\Auto

In [49]:
import glob, os, numpy as np
xy_files = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))
bad = []
for i,p in enumerate(xy_files):
    try:
        a = np.load(p)
    except Exception as e:
        bad.append((p, 'load_error', str(e)))
        continue
    ok = (a.ndim == 3 and a.shape[1:] == (17,2))
    if not ok:
        bad.append((p, 'shape', a.shape, a.size, a.ndim))

print('checked', len(xy_files), 'files, bad count =', len(bad))
for b in bad:
    print(b)

# show a quick stats of sizes
from collections import Counter
sizes = Counter()
for p in xy_files:
    a = np.load(p)
    sizes[a.size] += 1
print('\nunique sizes count:', len(sizes))
for s,cnt in sizes.most_common(10):
    print(s, cnt)

checked 60 files, bad count = 0

unique sizes count: 45
7242 4
7140 3
6630 3
7004 3
6460 3
6256 3
8806 2
6426 2
7276 1
7548 1


In [50]:
import glob, os, numpy as np
xy_files = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))
for p in xy_files[:6]:
    print('\nFILE:', p)
    a = np.load(p)
    print(' raw shape:', a.shape, 'size', a.size, 'ndim', a.ndim)
    try:
        T = a.shape[0]
        xy_n = _normalize_xy_by_shoulders(a)
        print(' after normalize shape:', xy_n.shape, 'size', xy_n.size, 'ndim', xy_n.ndim)
        try:
            xy_flat = xy_n.reshape(T, 34)
            print(' reshape OK ->', xy_flat.shape)
        except Exception as e:
            print(' reshape ERROR:', e)
    except Exception as e:
        print(' normalize ERROR:', e)



FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\10_xy.npy
 raw shape: (210, 17, 2) size 7140 ndim 3
 after normalize shape: (210, 17, 2) size 7140 ndim 3
 reshape OK -> (210, 34)

FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\11_xy.npy
 raw shape: (195, 17, 2) size 6630 ndim 3
 after normalize shape: (195, 17, 2) size 6630 ndim 3
 reshape OK -> (195, 34)

FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\12_xy.npy
 raw shape: (214, 17, 2) size 7276 ndim 3
 after normalize shape: (214, 17, 2) size 7276 ndim 3
 reshape OK -> (214, 34)

FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\13_xy.npy
 raw shape: (222, 17, 2) size 7548 ndim 3
 after normalize shape: (222, 17, 2) size 7548 ndim 3
 reshape OK -> (222, 34)

FILE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\14_xy.npy
 raw shape: (239, 17, 2) size 8126 ndim 3
 after normali

In [51]:
import numpy as np, glob, os
p = sorted(glob.glob(os.path.join(OUT_XY_TRAIN, "*_xy.npy")))[0]
print('file', p)
xy = np.load(p)
print('xy shape', xy.shape, 'dtype', xy.dtype)
xy_17 = np.asarray(xy, dtype=np.float32)
print('xy_17 shape', xy_17.shape, 'ndim', xy_17.ndim)
LS_POS, RS_POS = 11,12
left = xy_17[:, LS_POS, :]
right = xy_17[:, RS_POS, :]
print('left', left.shape, 'right', right.shape)
c = (left + right) / 2.0
s = np.linalg.norm(right - left, axis=1, keepdims=True)
print('c', c.shape, 's', s.shape)
print('c[:,None,:]', c[:,None,:].shape)
print('s[:,None,None]', s[:,None,None].shape)
res = (xy_17 - c[:, None, :])
print('after subtract shape', res.shape)
res2 = res / s[:, None, None]
print('after divide shape', res2.shape, 'dtype', res2.dtype)
print('res2 ndim', res2.ndim)
print('Res2 first element sample:', res2[0,0,:4])


file C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\npy_cikti\10_xy.npy
xy shape (210, 17, 2) dtype float32
xy_17 shape (210, 17, 2) ndim 3
left (210, 2) right (210, 2)
c (210, 2) s (210, 1)
c[:,None,:] (210, 1, 2)
s[:,None,None] (210, 1, 1, 1)
after subtract shape (210, 17, 2)
after divide shape (210, 210, 17, 2) dtype float32
res2 ndim 4
Res2 first element sample: [[-0.00978709 -1.0074695 ]
 [ 0.04215965 -1.1637623 ]
 [ 0.08086261 -1.1648535 ]
 [ 0.11329053 -1.1616557 ]]


# Adım 3 — TCN-Tabanlı Autoencoder (AE) Eğitimi ve Gömleme Çıkarma
Girdi: (B, C=40, T=30) → Encoder (TCN) → **embedding=64** → Decoder (TCN) → (B,40,30).
Kayıp: MSE (rekonstrüksiyon). En iyi model kaydedilir, sonra **train/test** için pencere gömlemeleri `.npy` olarak dışa aktarılır.


## 3.1 — Konfig, cihaz ve klasörler

In [53]:
import os, math, json, numpy as np, torch, torch.nn as nn
from torch.utils.data import DataLoader

# 2.7’de tanımladığımız değişkenleri kullanıyoruz:
# BASE_ROOT, WORK_DIR, STATS_JSON, CFG, train_loader, val_loader
# Ayrıca 2.7’deki yardımcıları ( _make_features_CT, _window_slice_or_pad, _load_feature_stats ) kullanacağız.

# Test dizinleri (Adım 2'de üretmiştik)
OUT_XY_TEST  = os.path.join(WORK_DIR, "npy_test_cikti")
OUT_ANG_TEST = os.path.join(WORK_DIR, "npy_aci_test_cikti")

MODEL_DIR = os.path.join(WORK_DIR, "models"); os.makedirs(MODEL_DIR, exist_ok=True)
EMB_DIR_TRAIN = os.path.join(WORK_DIR, "embeddings", "train"); os.makedirs(EMB_DIR_TRAIN, exist_ok=True)
EMB_DIR_TEST  = os.path.join(WORK_DIR, "embeddings", "test");  os.makedirs(EMB_DIR_TEST,  exist_ok=True)

# AE hiperparametrelerini CFG'ye ekle (varsa üzerine yazmaz)
CFG.setdefault("emb_dim", 64)
CFG.setdefault("hidden", 128)
CFG.setdefault("lr", 1e-3)
CFG.setdefault("weight_decay", 1e-4)
CFG.setdefault("max_epochs", 40)
CFG.setdefault("early_stop_patience", 6)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

## 3.2 — Model: TCN Encoder/Decoder (residual, dilated)

In [54]:
class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=1, k=3, p=None, dropout=0.1):
        super().__init__()
        if p is None: p = dilation * (k-1)//2  # length sabit kalsın
        self.conv1 = nn.Conv1d(in_ch, out_ch, k, padding=p, dilation=dilation)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.act1  = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(out_ch, out_ch, k, padding=p, dilation=dilation)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.act2  = nn.GELU()
        self.proj  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        # x: (B,C,T)
        y = self.conv1(x); y = self.bn1(y); y = self.act1(y); y = self.dropout(y)
        y = self.conv2(y); y = self.bn2(y)
        return self.act2(y + self.proj(x))

class TCNEncoder(nn.Module):
    def __init__(self, in_ch=40, hidden=128, emb_dim=64, dilations=(1,2,4,8)):
        super().__init__()
        layers = []
        ch = in_ch
        for d in dilations:
            layers.append(ResBlock1D(ch, hidden, dilation=d))
            ch = hidden
        self.net = nn.Sequential(*layers)
        self.head = nn.Conv1d(hidden, hidden, 1)
        self.pool = nn.AdaptiveAvgPool1d(1)   # (B,hidden,1)
        self.to_emb = nn.Linear(hidden, emb_dim)

    def forward(self, x):
        # x: (B,40,T)
        h = self.net(x)                # (B,hidden,T)
        h2 = self.head(h)              # (B,hidden,T)
        g = self.pool(h2).squeeze(-1)  # (B,hidden)
        z = self.to_emb(g)             # (B,emb_dim)
        return h2, z                   # zaman-özellik, embedding

class TCNDecoder(nn.Module):
    def __init__(self, out_ch=40, hidden=128, dilations=(8,4,2,1)):
        super().__init__()
        layers = []
        ch = hidden
        for d in dilations:
            layers.append(ResBlock1D(ch, hidden, dilation=d))
            ch = hidden
        self.net = nn.Sequential(*layers)
        self.out = nn.Conv1d(hidden, out_ch, 1)

    def forward(self, h_time):
        # h_time: (B,hidden,T)
        y = self.net(h_time)
        return self.out(y)  # (B,40,T)

class AE_TCN(nn.Module):
    def __init__(self, in_ch=40, hidden=128, emb_dim=64):
        super().__init__()
        self.encoder = TCNEncoder(in_ch=in_ch, hidden=hidden, emb_dim=emb_dim)
        self.decoder = TCNDecoder(out_ch=in_ch, hidden=hidden)

    @torch.no_grad()
    def encode(self, x):
        # x: (B,40,T)
        h_time, z = self.encoder(x)
        return z

    def forward(self, x):
        h_time, z = self.encoder(x)
        x_hat = self.decoder(h_time)
        return x_hat, z


## 3.3 — Eğitim döngüsü (early stopping + en iyi modeli kaydet)

In [55]:
def train_ae(model, train_loader, val_loader, cfg, model_dir):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    crit = nn.MSELoss()

    best_val = float("inf")
    patience = 0
    best_path = os.path.join(model_dir, "best_ae_tcn.pt")

    for ep in range(1, cfg["max_epochs"]+1):
        model.train()
        tr_loss, ntr = 0.0, 0
        for xb in train_loader:
            xb = xb.to(device)  # (B,40,30)
            x_hat, _ = model(xb)
            loss = crit(x_hat, xb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            tr_loss += loss.item() * xb.size(0); ntr += xb.size(0)

        model.eval()
        va_loss, nva = 0.0, 0
        with torch.no_grad():
            for xb in val_loader:
                xb = xb.to(device)
                x_hat, _ = model(xb)
                loss = crit(x_hat, xb)
                va_loss += loss.item() * xb.size(0); nva += xb.size(0)

        tr = tr_loss / max(1,ntr)
        va = va_loss / max(1,nva)
        print(f"Epoch {ep:02d}/{cfg['max_epochs']}  train={tr:.6f}  val={va:.6f}")

        if va < best_val - 1e-6:
            best_val = va
            patience = 0
            torch.save({"state_dict": model.state_dict(),
                        "cfg": cfg}, best_path)
            print("  ↳ best updated & saved:", best_path)
        else:
            patience += 1
            if patience >= cfg["early_stop_patience"]:
                print("  ↳ early stop.")
                break

    # En iyi modeli geri yükle
    if os.path.exists(best_path):
        ckpt = torch.load(best_path, map_location=device)
        model.load_state_dict(ckpt["state_dict"])
    return model, best_path


## 3.4 — Eğitimi başlat

In [56]:
model = AE_TCN(in_ch=40, hidden=CFG["hidden"], emb_dim=CFG["emb_dim"])
model, BEST_PATH = train_ae(model, train_loader, val_loader, CFG, MODEL_DIR)
print("Best model:", BEST_PATH)


Epoch 01/40  train=1.310158  val=0.635254
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 02/40  train=0.504361  val=0.585095
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 02/40  train=0.504361  val=0.585095
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 03/40  train=0.298549  val=0.391577
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 03/40  train=0.298549  val=0.391577
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\ae_stajdevam\models\best_ae_tcn.pt
Epoch 04/40  train=0.209853  val=0.244242
  ↳ best updated & saved: C:\Users\The Coder Farmer\Desktop\Autoencode

## 3.5 — Gömleme dışa aktarma (train/test → .npy)

In [57]:
from pathlib import Path
import glob

@torch.no_grad()
def export_embeddings(model, xy_dir, ang_dir, out_dir, cfg, file_list=None, stats_json=STATS_JSON):
    os.makedirs(out_dir, exist_ok=True)
    mean, std = _load_feature_stats(stats_json)

    xy_files = sorted(glob.glob(os.path.join(xy_dir, "*_xy.npy")))
    if file_list is not None:
        xy_files = [os.path.join(xy_dir, f"{b}_xy.npy") for b in file_list]
    assert xy_files, f"Boş dizin: {xy_dir}"

    win, stride, pad = cfg["window_len"], cfg["stride"], cfg["pad_mode"]
    bs = cfg["batch_size"]
    saved = []

    model.eval()

    for p in xy_files:
        base = Path(p).name.replace("_xy.npy","")
        pang = os.path.join(ang_dir, f"{base}_ang.npy")
        xy = np.load(p)
        ang = np.load(pang) if os.path.exists(pang) else None

        feat = _make_features_CT(xy, ang, cfg["use_angles"])   # (40,T)
        T = feat.shape[1]

        # window başlangıçları
        starts = [0] if T <= win else list(range(0, T - win + 1, stride))
        if T > win and (T - win) % stride != 0:
            starts.append(T - win)

        # mini-batch ile encode
        embs = []
        for i in range(0, len(starts), bs):
            batch_starts = starts[i:i+bs]
            x_list = []
            for st in batch_starts:
                xw = _window_slice_or_pad(feat, st, win, pad)        # (40,win)
                xw = (xw - mean[:,None]) / (std[:,None] + 1e-8)      # standardize
                x_list.append(xw)
            x = np.stack(x_list, axis=0)                              # (B,40,win)
            x = torch.from_numpy(x).float().to(device)
            z = model.encode(x)                                       # (B,emb_dim)
            embs.append(z.cpu().numpy())

        E = np.concatenate(embs, axis=0) if embs else np.zeros((0, cfg["emb_dim"]), np.float32)
        out_path = os.path.join(out_dir, f"{base}_emb.npy")
        np.save(out_path, E.astype(np.float32))
        saved.append((base, E.shape))
    return saved

# Train emb'leri
train_saved = export_embeddings(model,
                                xy_dir=os.path.join(WORK_DIR, "npy_cikti"),
                                ang_dir=os.path.join(WORK_DIR, "npy_aci_cikti"),
                                out_dir=EMB_DIR_TRAIN,
                                cfg=CFG)
print("Train embeddings saved:", len(train_saved), "dosya")

# Test emb'leri (61,62,63)
test_saved = export_embeddings(model,
                               xy_dir=OUT_XY_TEST,
                               ang_dir=OUT_ANG_TEST,
                               out_dir=EMB_DIR_TEST,
                               cfg=CFG)
print("Test embeddings saved:", len(test_saved), "dosya")


Train embeddings saved: 60 dosya
Test embeddings saved: 3 dosya
Test embeddings saved: 3 dosya


# Adım 4 — DTW-yol Ortalama Cosine (%) ile Benzerlik
Karşılaştırmalar:
- 61 ↔ 61 (kontrol)
- 61 ↔ 62 (farklı hareket)
- 61 ↔ 63 (aynı hareket, **2× hız**) 


## 4.1 — Yardımcılar: gömleme yükleme + DTW yol ve skor

In [58]:
import os, numpy as np

# Gömleme klasörleri (Adım 3'te oluşturduk)
EMB_DIR_TEST  = os.path.join(WORK_DIR, "embeddings", "test")
EMB_DIR_TRAIN = os.path.join(WORK_DIR, "embeddings", "train")  # gerekirse

# fastdtw varsa kullan, yoksa klasik DTW (path’li) kullan
try:
    from fastdtw import fastdtw
    _HAVE_FASTDTW = True
except Exception:
    _HAVE_FASTDTW = False

def load_emb(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Bulunamadı: {path}")
    arr = np.load(path)
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2:
        arr = arr.reshape(arr.shape[0], -1)
    return arr  # (T, D) = (Nwin, emb_dim)

def _cosine_distance(u: np.ndarray, v: np.ndarray, eps: float=1e-8) -> float:
    uu = float(np.linalg.norm(u) + eps)
    vv = float(np.linalg.norm(v) + eps)
    return 1.0 - float(np.dot(u, v) / (uu * vv))

def _dtw_path_exact(A: np.ndarray, B: np.ndarray):
    # Klasik DTW — path döndürür (T1*T2 karmaşıklık)
    T1, D1 = A.shape; T2, D2 = B.shape
    assert D1 == D2, f"embed boyutu farklı: {D1} vs {D2}"
    C = np.full((T1+1, T2+1), np.inf, dtype=np.float64)
    C[0,0] = 0.0
    back = np.zeros((T1+1, T2+1, 2), dtype=np.int32) - 1
    for i in range(1, T1+1):
        for j in range(1, T2+1):
            cost = _cosine_distance(A[i-1], B[j-1])
            # üç komşudan en düşüğü
            k = np.argmin((C[i-1,j], C[i,j-1], C[i-1,j-1]))
            if k == 0:   C[i,j] = cost + C[i-1,j];   back[i,j] = (i-1, j)
            elif k == 1: C[i,j] = cost + C[i, j-1];  back[i,j] = (i, j-1)
            else:        C[i,j] = cost + C[i-1,j-1]; back[i,j] = (i-1, j-1)
    # geri izle
    path = []
    i, j = T1, T2
    while not (i == 0 and j == 0):
        path.append((i-1, j-1))
        i, j = back[i,j]
    path.reverse()
    return path

def dtw_mean_cosine_percent(E1: np.ndarray, E2: np.ndarray) -> float:
    """
    E1, E2: (T, D) gömleme dizileri
    Dönüş: DTW yolundaki ortalama cosine similarity * 100
    """
    E1 = np.asarray(E1, dtype=np.float32)
    E2 = np.asarray(E2, dtype=np.float32)
    assert E1.ndim == 2 and E2.ndim == 2
    assert E1.shape[1] == E2.shape[1], f"embed boyutu uymuyor: {E1.shape[1]} vs {E2.shape[1]}"

    if _HAVE_FASTDTW:
        # fastdtw path döndürür
        _, path = fastdtw(E1, E2, dist=_cosine_distance)
    else:
        path = _dtw_path_exact(E1, E2)

    sims = []
    for i, j in path:
        sims.append(1.0 - _cosine_distance(E1[i], E2[j]))
    mean_sim = float(np.mean(sims)) if sims else 0.0
    return 100.0 * mean_sim


## 4.2 — 61/62/63 karşılaştırmaları

In [59]:
# Dosya yolları
p61 = os.path.join(EMB_DIR_TEST, "61_emb.npy")
p62 = os.path.join(EMB_DIR_TEST, "62_emb.npy")
p63 = os.path.join(EMB_DIR_TEST, "63_emb.npy")  # 61'in 2× hızı

# Yükle
E61 = load_emb(p61)
E62 = load_emb(p62)
E63 = load_emb(p63)

# Skorlar
sim_61_61 = dtw_mean_cosine_percent(E61, E61)   # kontrol
sim_61_62 = dtw_mean_cosine_percent(E61, E62)   # farklı hareket
sim_61_63 = dtw_mean_cosine_percent(E61, E63)   # aynı hareket 2× hız

print(f"61 vs 61  (kontrol)   : {sim_61_61:6.2f} %")
print(f"61 vs 62  (farklı)    : {sim_61_62:6.2f} %")
print(f"61 vs 63  (aynı, 2×)  : {sim_61_63:6.2f} %")


61 vs 61  (kontrol)   : 100.00 %
61 vs 62  (farklı)    :  52.46 %
61 vs 63  (aynı, 2×)  :  94.90 %


In [61]:
import os, numpy as np, torch, torch.nn as nn

def recon_mse_for_video(vid):
    xy_p  = os.path.join(WORK_DIR, "npy_test_cikti", f"{vid}_xy.npy")
    ang_p = os.path.join(WORK_DIR, "npy_aci_test_cikti", f"{vid}_ang.npy")
    xy = np.load(xy_p); ang = np.load(ang_p) if os.path.exists(ang_p) else None
    feat = _make_features_CT(xy, ang, CFG["use_angles"])                    # (40,T)
    mean, std = _load_feature_stats(STATS_JSON)
    win, stride, pad = CFG["window_len"], CFG["stride"], CFG["pad_mode"]

    # pencereleri topla
    T = feat.shape[1]
    starts = [0] if T<=win else list(range(0, T-win+1, stride))
    if T>win and (T-win)%stride!=0: starts.append(T-win)
    X = []
    for st in starts:
        xw = _window_slice_or_pad(feat, st, win, pad)
        xw = (xw - mean[:,None])/(std[:,None]+1e-8)
        X.append(xw)
    X = torch.from_numpy(np.stack(X)).float().to(device)                     # (B,40,30)

    model.eval()
    with torch.no_grad():
        x_hat, _ = model(X)
        mse = nn.functional.mse_loss(x_hat, X, reduction="mean").item()
    return mse

for v in [61,62,63]:
    print(v, "MSE:", recon_mse_for_video(v))


61 MSE: 0.09801687300205231
62 MSE: 61.76719284057617
63 MSE: 0.13490988314151764
